In [1]:
!pip install optuna

In [16]:
import os
import numpy as np
import pandas as pd

In [17]:
import gdown
import optuna
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error,mean_squared_error,mean_absolute_percentage_error as mape
from sklearn.linear_model import Ridge

tscv = TimeSeriesSplit(n_splits=3)

In [18]:
import warnings
warnings.filterwarnings('ignore')

In [19]:
datafile_dict={
    "train_transformed_data.csv": "1w_ZOLTTc8WnLj7l8bnwo8Vuz1ii_ZBTY"
    ,"validate_transformed_data.csv" : "1SI5pS8igjgATw5fSPB8hYt5vc62aiVHF"
    ,"test_data.csv" : "1JhzqIAupBjD-vZTqnaAqL1OI9HIpD9O6"
}
#remove existing files
for filename in datafile_dict.keys():
    if os.path.exists(filename):
        os.remove(filename)
        print(f"Removed {filename}")
    else:
        print(f"{filename} not found, skipping removal.")
#doanload new files from drive
for filename, file_id in datafile_dict.items():
  gdown.download(id=file_id, output=filename, quiet=False)

Removed train_transformed_data.csv
Removed validate_transformed_data.csv
Removed test_data.csv


Downloading...
From: https://drive.google.com/uc?id=1w_ZOLTTc8WnLj7l8bnwo8Vuz1ii_ZBTY
To: /content/train_transformed_data.csv
100%|██████████| 28.5M/28.5M [00:00<00:00, 190MB/s]
Downloading...
From: https://drive.google.com/uc?id=1SI5pS8igjgATw5fSPB8hYt5vc62aiVHF
To: /content/validate_transformed_data.csv
100%|██████████| 6.29M/6.29M [00:00<00:00, 123MB/s]
Downloading...
From: https://drive.google.com/uc?id=1JhzqIAupBjD-vZTqnaAqL1OI9HIpD9O6
To: /content/test_data.csv
100%|██████████| 572k/572k [00:00<00:00, 87.2MB/s]


In [20]:
def load_data_dat(filename):
  if filename.endswith('.csv'):
    df=pd.read_csv(filename)
  return df

train_df= load_data_dat('train_transformed_data.csv')
val_df= load_data_dat('validate_transformed_data.csv')
test_df= load_data_dat('test_data.csv')

In [21]:
FEATURES = [
    'Store_id',#'Store_Type','Location_Type','Region_Code',
    'Store_Type_S1',
       'Store_Type_S2', 'Store_Type_S3', 'Store_Type_S4', 'Location_Type_L1',
       'Location_Type_L2', 'Location_Type_L3', 'Location_Type_L4',
       'Location_Type_L5', 'Region_Code_R1', 'Region_Code_R2',
       'Region_Code_R3', 'Region_Code_R4',
    'Discount_Flag','Holiday',
    'day','month','weekday','weekofyear','is_weekend',
    'lag_1_#Order','lag_7_#Order','rolling_#Order_7',
    'lag_1_Sales','lag_7_Sales','rolling_Sales_7',
    'lag_1_aov',
    'Sales','#Order'
]
FEATURES_ORDERS = [
    'Store_id',#'Store_Type','Location_Type','Region_Code',
    'Store_Type_S1',
       'Store_Type_S2', 'Store_Type_S3', 'Store_Type_S4', 'Location_Type_L1',
       'Location_Type_L2', 'Location_Type_L3', 'Location_Type_L4',
       'Location_Type_L5', 'Region_Code_R1', 'Region_Code_R2',
       'Region_Code_R3', 'Region_Code_R4',
    'Discount_Flag','Holiday',
    'day','month','weekday','weekofyear','is_weekend',
    'lag_1_#Order','lag_7_#Order','rolling_#Order_7',
    'lag_1_Sales','lag_7_Sales','rolling_Sales_7',
    'lag_1_aov'
]

FEATURES_SALES = [
    'Store_id',#'Store_Type','Location_Type','Region_Code',
    'Store_Type_S1',
       'Store_Type_S2', 'Store_Type_S3', 'Store_Type_S4', 'Location_Type_L1',
       'Location_Type_L2', 'Location_Type_L3', 'Location_Type_L4',
       'Location_Type_L5', 'Region_Code_R1', 'Region_Code_R2',
       'Region_Code_R3', 'Region_Code_R4',
    'Discount_Flag','Holiday',
    'day','month','weekday','weekofyear','is_weekend',
    'lag_1_Sales','lag_7_Sales','rolling_Sales_7',
    'lag_1_#Order','lag_7_#Order','rolling_#Order_7',
    'lag_1_aov','pred_orders'
]

column_dtype_map_dict = {'Store_id': 'int64',
 'Store_Type_S1': 'int64',
 'Store_Type_S2': 'int64',
 'Store_Type_S3': 'int64',
 'Store_Type_S4': 'int64',
 'Location_Type_L1': 'int64',
 'Location_Type_L2': 'int64',
 'Location_Type_L3': 'int64',
 'Location_Type_L4': 'int64',
 'Location_Type_L5': 'int64',
 'Region_Code_R1': 'int64',
 'Region_Code_R2': 'int64',
 'Region_Code_R3': 'int64',
 'Region_Code_R4': 'int64',
 'Discount_Flag': 'int64',
 'Holiday': 'int64',
 'day': 'int64',
 'month': 'int64',
 'weekday': 'int64',
 'weekofyear': 'int64',
 'is_weekend': 'int64',
 'lag_1_#Order': 'float64',
 'lag_7_#Order': 'float64',
 'rolling_#Order_7': 'float64',
 'lag_1_Sales': 'float64',
 'lag_7_Sales': 'float64',
 'rolling_Sales_7': 'float64',
 'lag_1_aov': 'float64',
 'Sales': 'float64',
 '#Order': 'float64',
 'pred_orders': 'float64'}

In [22]:
train_df['Date'] = pd.to_datetime(train_df['Date'])
val_df['Date'] = pd.to_datetime(val_df['Date'])
train_df = train_df.sort_values("Date")
val_df = val_df.sort_values("Date")
train_df=train_df[FEATURES]
val_df=val_df[FEATURES]

In [23]:
def format_df_dtype(df, features_dtype_map_dict):
    for col, dtype in features_dtype_map_dict.items():
        if col in df.columns:
            df[col] = df[col].astype(dtype)
    return df

train_df = format_df_dtype(train_df, column_dtype_map_dict)
val_df = format_df_dtype(val_df, column_dtype_map_dict)

#**Ridge**
---

##**1. Orders**
---

In [24]:
TARGET = "#Order"  # or "#Order"
train_df2=train_df.copy(deep=True).dropna()
X = train_df2[FEATURES_ORDERS]
y = train_df2[TARGET]


In [41]:
def wape(y_true, y_pred):
    return np.sum(np.abs(y_true - y_pred)) / (np.sum(y_true) + 1e-8)

In [26]:
tscv = TimeSeriesSplit(n_splits=3)
mean_metric_scores = []
def objective(trial):

    params = {
        "alpha": trial.suggest_float("alpha", 0.1, 50)
    }

    wape_scores = []
    mae_scores = []
    mse_scores = []
    rmse_scores = []
    mape_scores = []

    for train_idx, val_idx in tscv.split(X):

        X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
        y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]

        model = Ridge(**params)
        model.fit(X_train_fold, y_train_fold)

        preds = model.predict(X_val_fold)
        #score = mean_absolute_error(y_val_fold, preds)
        #score = wape(y_val_fold, preds)

        wape_scores.append(wape(y_val_fold, preds))
        mae_scores.append(mean_absolute_error(y_val_fold, preds))
        mse_scores.append(mean_squared_error(y_val_fold, preds))
        rmse_scores.append(np.sqrt(mean_squared_error(y_val_fold, preds)))
        mape_scores.append(mape(y_val_fold, preds))

    average_metric = np.mean(mae_scores)
    mean_score = mean_metric_scores.append((np.mean(wape_scores),np.mean(mae_scores),np.mean(mse_scores),np.mean(rmse_scores),np.mean(mape_scores),params))
    return average_metric

In [27]:

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=30)

print("Best Params:", study.best_params)
print("Best Score:", study.best_value)

[I 2026-04-20 19:56:02,844] A new study created in memory with name: no-name-23f2fe6a-709b-4323-aaa4-51cc32ed68f5
[I 2026-04-20 19:56:03,002] Trial 0 finished with value: 10.580634438659338 and parameters: {'alpha': 18.13636412778741}. Best is trial 0 with value: 10.580634438659338.
[I 2026-04-20 19:56:03,140] Trial 1 finished with value: 10.65858717002159 and parameters: {'alpha': 5.407462241711664}. Best is trial 0 with value: 10.580634438659338.
[I 2026-04-20 19:56:03,278] Trial 2 finished with value: 10.660814733612437 and parameters: {'alpha': 5.127316596971664}. Best is trial 0 with value: 10.580634438659338.
[I 2026-04-20 19:56:03,422] Trial 3 finished with value: 10.554266257414783 and parameters: {'alpha': 24.143467130042822}. Best is trial 3 with value: 10.554266257414783.
[I 2026-04-20 19:56:03,593] Trial 4 finished with value: 10.548814670195325 and parameters: {'alpha': 25.529628685653346}. Best is trial 4 with value: 10.548814670195325.
[I 2026-04-20 19:56:03,734] Trial 5

Best Params: {'alpha': 49.927017883207526}
Best Score: 10.477927666749991


In [28]:
formatted_data = []
for wape, mae, mse, rmse, mape, params in mean_metric_scores:
    # Combine metrics and the params dictionary into one flat dict
    row = {
        'WAPE': wape, 'MAE': mae, 'MSE': mse, 
        'RMSE': rmse, 'MAPE': mape
    }
    row.update(params)
    formatted_data.append(row)

df = pd.DataFrame(formatted_data)
df

,WAPE,MAE,MSE,RMSE,MAPE,alpha
0,0.155943,10.580634,240.409418,15.338004,3.830445e+13,18.136364
1,0.157100,10.658587,251.015777,15.625077,3.839975e+13,5.407462
2,0.157133,10.660815,251.330368,15.633427,3.840259e+13,5.127317
3,0.155552,10.554266,237.114504,15.246389,3.827468e+13,24.143467
4,0.155471,10.548815,236.458199,15.227988,3.826872e+13,25.529629
5,0.155022,10.518549,232.988666,15.129834,3.823686e+13,34.317243
6,0.155290,10.536642,235.026396,15.187664,3.825564e+13,28.828440
7,0.154866,10.508071,231.861389,15.097612,3.822635e+13,37.861393
8,0.156128,10.593062,242.024345,15.382460,3.831896e+13,15.657871
9,0.155684,10.563186,238.207410,15.276916,3.828459e+13,21.987626


In [29]:
df.to_csv('ridge_errorfile_orders.csv', header=True, index=False)

In [30]:
study.best_params

{'alpha': 49.927017883207526}

##**2. Sales**
---

In [42]:
train_df= load_data_dat('train_transformed_data.csv')
val_df= load_data_dat('validate_transformed_data.csv')
test_df= load_data_dat('test_data.csv')

In [43]:
train_df['Date'] = pd.to_datetime(train_df['Date'])
val_df['Date'] = pd.to_datetime(val_df['Date'])
train_df = train_df.sort_values("Date")
val_df = val_df.sort_values("Date")
train_df=train_df[FEATURES]
val_df=val_df[FEATURES]

In [44]:
def format_df_dtype(df, features_dtype_map_dict):
    for col, dtype in features_dtype_map_dict.items():
        if col in df.columns:
            df[col] = df[col].astype(dtype)
    return df

train_df = format_df_dtype(train_df, column_dtype_map_dict)
val_df = format_df_dtype(val_df, column_dtype_map_dict)

In [45]:
TARGET = "#Order"  # or "#Order"
train_df2=train_df.copy(deep=True).dropna()
X = train_df2[FEATURES_ORDERS]
y = train_df2[TARGET]


In [53]:
order_params = {
            'alpha': 49.927017883207526
            }
model_orders = Ridge(**order_params)

model_orders.fit(X, y)
X['pred_orders'] = model_orders.predict(X)

In [54]:
X[X['Store_id'] == 1].head(10)

,Store_id,Store_Type_S1,Store_Type_S2,Store_Type_S3,Store_Type_S4,Location_Type_L1,Location_Type_L2,Location_Type_L3,Location_Type_L4,Location_Type_L5,...,weekofyear,is_weekend,lag_1_#Order,lag_7_#Order,rolling_#Order_7,lag_1_Sales,lag_7_Sales,rolling_Sales_7,lag_1_aov,pred_orders
7,1,1,0,0,0,0,0,1,0,0,...,2,0,55.0,9.0,51.285714,36855.0,7011.84,36109.834286,658.125000,30688.246189
8,1,1,0,0,0,0,0,1,0,0,...,2,0,52.0,60.0,57.428571,34101.0,42369.00,39979.714286,643.415094,29898.437639
9,1,1,0,0,0,0,0,1,0,0,...,2,0,65.0,72.0,58.142857,42429.0,50037.00,39988.285714,642.863636,30960.646132
10,1,1,0,0,0,0,0,1,0,0,...,2,0,53.0,64.0,55.428571,33510.0,44397.00,37627.285714,620.555556,28097.276646
11,1,1,0,0,0,0,0,1,0,0,...,2,0,56.0,63.0,54.285714,36873.0,47604.00,36552.428571,646.894737,28662.406180
12,1,1,0,0,0,0,0,1,0,0,...,2,1,35.0,36.0,50.285714,25644.0,24495.00,33415.285714,712.333333,46553.634931
13,1,1,0,0,0,0,0,1,0,0,...,2,1,57.0,55.0,53.285714,40890.0,36855.00,35757.428571,705.000000,40655.018561
14,1,1,0,0,0,0,0,1,0,0,...,3,0,56.0,52.0,53.428571,39254.4,34101.00,36100.200000,688.673684,41836.686074
15,1,1,0,0,0,0,0,1,0,0,...,3,0,78.0,65.0,57.142857,51993.0,42429.00,38656.200000,658.139241,44855.883471
16,1,1,0,0,0,0,0,1,0,0,...,3,0,64.0,53.0,57.000000,37716.0,33510.00,37982.914286,580.246154,41705.551210


In [55]:
TARGET = 'Sales'
y = train_df2[TARGET]

In [61]:
tscv = TimeSeriesSplit(n_splits=3)
mean_metric_scores = []
def objective(trial):

    params = {
        "alpha": trial.suggest_float("alpha", 0.1, 50)
    }

    wape_scores = []
    mae_scores = []
    mse_scores = []
    rmse_scores = []
    mape_scores = []

    for train_idx, val_idx in tscv.split(X):

        X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
        y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]

        model = Ridge(**params)
        model.fit(X_train_fold, y_train_fold)

        preds = model.predict(X_val_fold)
        #score = mean_absolute_error(y_val_fold, preds)
        #score = wape(y_val_fold, preds)

        wape_scores.append(wape(y_val_fold, preds))
        mae_scores.append(mean_absolute_error(y_val_fold, preds))
        mse_scores.append(mean_squared_error(y_val_fold, preds))
        rmse_scores.append(np.sqrt(mean_squared_error(y_val_fold, preds)))
        #mape_scores.append(mape(y_val_fold, preds))

    average_metric = np.mean(mae_scores)
    mean_score = mean_metric_scores.append((
                                np.mean(wape_scores), 
                                np.mean(mae_scores), 
                                np.mean(mse_scores), 
                                np.mean(rmse_scores), 
                                0,  # Placeholder for MAPE since it's commented out
                                params
                            ))
    return average_metric

In [62]:
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=30)

print("Best Params:", study.best_params)
print("Best Score:", study.best_value)

[I 2026-04-20 20:05:17,485] A new study created in memory with name: no-name-d7fcf63c-a850-4399-a3d0-e60a699d03fe
[I 2026-04-20 20:05:17,617] Trial 0 finished with value: 7046.57210582799 and parameters: {'alpha': 17.22132046935856}. Best is trial 0 with value: 7046.57210582799.
[I 2026-04-20 20:05:17,744] Trial 1 finished with value: 7036.670263349137 and parameters: {'alpha': 26.100968285537654}. Best is trial 1 with value: 7036.670263349137.
[I 2026-04-20 20:05:17,887] Trial 2 finished with value: 7038.624672193405 and parameters: {'alpha': 24.02914787778521}. Best is trial 1 with value: 7036.670263349137.
[I 2026-04-20 20:05:18,028] Trial 3 finished with value: 7066.5289348481165 and parameters: {'alpha': 6.894337679137302}. Best is trial 1 with value: 7036.670263349137.
[I 2026-04-20 20:05:18,158] Trial 4 finished with value: 7024.375018699368 and parameters: {'alpha': 45.175758977385755}. Best is trial 4 with value: 7024.375018699368.
[I 2026-04-20 20:05:18,289] Trial 5 finished 

Best Params: {'alpha': 49.96758009381003}
Best Score: 7022.291486194093


In [63]:
formatted_data = []
for wape, mae, mse, rmse, mape, params in mean_metric_scores:
    # Combine metrics and the params dictionary into one flat dict
    row = {
        'WAPE': wape, 'MAE': mae, 'MSE': mse, 
        'RMSE': rmse, 'MAPE': mape
    }
    row.update(params)
    formatted_data.append(row)

df = pd.DataFrame(formatted_data)
df

,WAPE,MAE,MSE,RMSE,MAPE,alpha
0,0.164561,7046.572106,1.001582e+08,9931.404415,0,17.221320
1,0.164321,7036.670263,1.000472e+08,9924.960239,0,26.100968
2,0.164369,7038.624672,1.000691e+08,9926.237594,0,24.029148
3,0.165040,7066.528935,1.003855e+08,9944.342360,0,6.894338
4,0.164022,7024.375019,9.990895e+07,9916.846825,0,45.175759
5,0.164304,7035.963441,1.000392e+08,9924.497402,0,26.898490
6,0.164166,7030.267918,9.997520e+07,9920.747823,0,34.467801
7,0.164363,7038.390742,1.000665e+08,9926.085204,0,24.266855
8,0.165180,7072.436509,1.004539e+08,9948.166257,0,4.914237
9,0.164275,7034.779836,1.000259e+08,9923.720915,0,28.297200


In [64]:
df.to_csv('ridge_errorfile_sales.csv', header=True, index=False)

In [65]:
study.best_params

{'alpha': 49.96758009381003}